In [ ]:
%gui qt
%load_ext autoreload
%autoreload 2

In [ ]:
import napari
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import napari
from sklearn.cluster import DBSCAN
import hmt_functions as hmt

In [ ]:
me3_raw_df = pd.read_csv("test_data/k27_k27_thaw009_me3.csv")
ac_raw_df = pd.read_csv("test_data/k27_k27_thaw009_ac.csv")

In [ ]:
me3_filtered_df = hmt.filter_axial(me3_raw_df)
ac_filtered_df = hmt.filter_axial(ac_raw_df)

In [ ]:
binary_mask, me3_df, ac_df = hmt.binarize_nucleus(me3_filtered_df, ac_filtered_df, thresh=2, show_plots=True)

In [ ]:
distance_map, contour_bands = hmt.create_radial_contours(binary_mask, show_plots=True)

In [ ]:
# =====================================================================
# --- SIMULATION PIPELINE ---
# =====================================================================

# 4. Extract parameters from the real data
print("Extracting me3 empirical parameters...")
me3_density_profile, me3_hists, me3_z_deg, x_min, y_min = hmt.extract_empirical_parameters(
    real_df=me3_df,                 # The masked data for the KD-Tree
    raw_me3_df=me3_filtered_df,     # Unmasked data for the global bounding box
    raw_ac_df=ac_filtered_df,       # Unmasked data for the global bounding box
    contour_bands=contour_bands, 
    bin_size=50,                    # Ensure this matches the bin_size from binarize_nucleus
    sdis=200, 
    step=10
)

print("Extracting ac empirical parameters...")
ac_density_profile, ac_hists, ac_z_deg, _, _ = hmt.extract_empirical_parameters(
    real_df=ac_df,                  # The masked data for the KD-Tree
    raw_me3_df=me3_filtered_df,     # Unmasked data for the global bounding box
    raw_ac_df=ac_filtered_df,       # Unmasked data for the global bounding box
    contour_bands=contour_bands, 
    bin_size=50,                    # Ensure this matches the bin_size from binarize_nucleus
    sdis=200, 
    step=10
)

In [ ]:
# 5. Build the probability map (The Blueprint)
print("Building 3D probability map...")
me3_prob_map = hmt.build_probability_map(distance_map, me3_density_profile)
ac_prob_map = hmt.build_probability_map(distance_map, ac_density_profile)

# 6. Plant the seeds (Monte Carlo Sampling)
# scaling_factor controls the overall density. Increase it for more seeds.
print("Planting seeds...")
me3_seed_coords = hmt.plant_seeds(
    prob_map=me3_prob_map, 
    real_z_coords=me3_df["z [nm]"].values, # Use real Z data for realistic axial distribution
    x_min=x_min, 
    y_min=y_min, 
    px_size=50.0, 
    scaling_factor=0.05
)
print(f"Planted {len(me3_seed_coords)} H3K27me3 seeds.")

ac_seed_coords = hmt.plant_seeds(
    prob_map=ac_prob_map, 
    real_z_coords=ac_df["z [nm]"].values,  # Use real Z data for realistic axial distribution
    x_min=x_min, 
    y_min=y_min, 
    px_size=50.0, 
    scaling_factor=0.05
)
print(f"Planted {len(ac_seed_coords)} H3K27ac seeds.")

# 7. Spawn the Nanodomains
print("Spawning nanodomains...")
simulated_me3_df = hmt.spawn_nanodomains(me3_seed_coords, me3_hists, me3_z_deg, sdis=200, step=10)
print(f"Generated {len(simulated_me3_df)} total simulated H3K27me3 localizations.")

simulated_ac_df = hmt.spawn_nanodomains(ac_seed_coords, ac_hists, ac_z_deg, sdis=200, step=10)
print(f"Generated {len(simulated_ac_df)} total simulated H3K27ac localizations.")

In [ ]:
density_samples = hmt.sample_lower_densities(me3_df, ac_df, show_plots=True, num_plots=8)

In [ ]:
# 1. Generate 10 subsampled fractions of your REAL experimental data
print("Subsampling real data...")
real_sampled_data = hmt.sample_lower_densities(me3_df, ac_df, num_samples=10, show_plots=False)
me3_real_fractions = [data[0] for data in real_sampled_data]

# 2. Setup the "Blueprint" once
me3_density_profile, me3_hists, me3_z_deg, x_min, y_min = hmt.extract_empirical_parameters(
    real_df=me3_df, raw_me3_df=me3_filtered_df, raw_ac_df=ac_filtered_df, contour_bands=contour_bands
)
me3_prob_map = hmt.build_probability_map(distance_map, me3_density_profile)

# Define your base scaling factor for 100% density
base_scaling_factor = 0.05 

# 3. Loop through each fraction and run a unique simulation
print("Running independent simulations using original plant_seeds...")
me3_gt_targets = []
size_param = 'bb_2D'

for i in range(1, 11):
    # Calculate the fraction multiplier (0.1, 0.2, ... 1.0)
    fraction_val = i / 10.0
    current_scaling = base_scaling_factor * fraction_val
    
    # Use the original plant_seeds function
    current_seeds = hmt.plant_seeds(
        me3_prob_map, 
        real_z_coords=me3_df["z [nm]"].values, 
        x_min=x_min, 
        y_min=y_min, 
        px_size=50.0, 
        scaling_factor=current_scaling
    )
    
    # Run the full 3D simulation on these seeds
    sim_frac_df = hmt.spawn_nanodomains(current_seeds, me3_hists, me3_z_deg)
    
    # Calculate the Ground Truth Size (e.g., Hull 3D)
    sizes = []
    for _, group_df in sim_frac_df.groupby("cluster_label"):
        res = hmt.calc_nanodomain_size(group_df)
        sizes.append(res[size_param])
        
    current_gt_size = np.mean(sizes) if sizes else 0
    me3_gt_targets.append(current_gt_size)
    
    print(f"Fraction {i}: Scaling {current_scaling:.4f} -> Seeds: {len(current_seeds)} -> GT Size: {current_gt_size:.2f}")

In [ ]:
# 4. Run the Epsilon Optimization
print("\nOptimizing Epsilon Parameters...")
nuclear_area_nm2 = np.sum(binary_mask) * (50 ** 2)

me3_coef, me3_exp = hmt.optimize_epsilon_power_law(
    fraction_dfs=me3_real_fractions, 
    gt_targets=me3_gt_targets, 
    nuclear_area_nm2=nuclear_area_nm2,
    size_metric=size_param,
    min_samples=8
)

print(f"\nME3 Equation: Epsilon = {me3_coef:.2f} * (Density)^{me3_exp:.4f}")

In [ ]:
sample_me3_df = density_samples[5][0]
sample_ac_df = density_samples[5][1]

In [ ]:
# =====================================================================
# --- SIMULATION PIPELINE ---
# =====================================================================

# 4. Extract parameters from the real data
print("Extracting me3 empirical parameters...")
me3_density_profile, me3_hists, me3_z_deg, x_min, y_min = hmt.extract_empirical_parameters(
    real_df=sample_me3_df,                 # The masked data for the KD-Tree
    raw_me3_df=me3_filtered_df,     # Unmasked data for the global bounding box
    raw_ac_df=ac_filtered_df,       # Unmasked data for the global bounding box
    contour_bands=contour_bands, 
    bin_size=50,                    # Ensure this matches the bin_size from binarize_nucleus
    sdis=200, 
    step=10
)

print("Extracting ac empirical parameters...")
ac_density_profile, ac_hists, ac_z_deg, _, _ = hmt.extract_empirical_parameters(
    real_df=sample_ac_df,                  # The masked data for the KD-Tree
    raw_me3_df=me3_filtered_df,     # Unmasked data for the global bounding box
    raw_ac_df=ac_filtered_df,       # Unmasked data for the global bounding box
    contour_bands=contour_bands, 
    bin_size=50,                    # Ensure this matches the bin_size from binarize_nucleus
    sdis=200, 
    step=10
)

In [ ]:
# 5. Build the probability map (The Blueprint)
print("Building 3D probability map...")
me3_prob_map = hmt.build_probability_map(distance_map, me3_density_profile)
ac_prob_map = hmt.build_probability_map(distance_map, ac_density_profile)

# 6. Plant the seeds (Monte Carlo Sampling)
# scaling_factor controls the overall density. Increase it for more seeds.
print("Planting seeds...")
me3_seed_coords = hmt.plant_seeds(
    prob_map=me3_prob_map, 
    real_z_coords=sample_me3_df["z [nm]"].values, # Use real Z data for realistic axial distribution
    x_min=x_min, 
    y_min=y_min, 
    px_size=50.0, 
    scaling_factor=0.05
)
print(f"Planted {len(me3_seed_coords)} H3K27me3 seeds.")

ac_seed_coords = hmt.plant_seeds(
    prob_map=ac_prob_map, 
    real_z_coords=sample_ac_df["z [nm]"].values,  # Use real Z data for realistic axial distribution
    x_min=x_min, 
    y_min=y_min, 
    px_size=50.0, 
    scaling_factor=0.05
)
print(f"Planted {len(ac_seed_coords)} H3K27ac seeds.")

# 7. Spawn the Nanodomains
print("Spawning nanodomains...")
simulated_me3_df = hmt.spawn_nanodomains(me3_seed_coords, me3_hists, me3_z_deg, sdis=200, step=10)
print(f"Generated {len(simulated_me3_df)} total simulated H3K27me3 localizations.")

simulated_ac_df = hmt.spawn_nanodomains(ac_seed_coords, ac_hists, ac_z_deg, sdis=200, step=10)
print(f"Generated {len(simulated_ac_df)} total simulated H3K27ac localizations.")

In [ ]:
hmt.plot_clusters_napari(simulated_me3_df, simulated_ac_df)

In [ ]:
sim_me3_old = pd.read_csv("C:/Users/georg/Downloads/simdens/simdens10surround2_me3.csv")
sim_ac_old = pd.read_csv("C:/Users/georg/Downloads/simdens/simdens10surround2_ac.csv")

sim_me3_old['sigmax [nm]'] = 110.0 
sim_me3_old['cluster_label'] = 1 
sim_me3_old['cluster_color'] = [(0, 1, 0, 0.5)] * len(sim_me3_old) # Green

sim_ac_old['sigmax [nm]'] = 110.0 
sim_ac_old['cluster_label'] = 1 
sim_ac_old['cluster_color'] = [(1, 0, 0, 0.5)] * len(sim_ac_old) # Red

hmt.plot_clusters_napari(sim_me3_old, sim_ac_old)

In [ ]:
me3_df = hmt.cluster_dbscan(me3_df, eps=50)
ac_df = hmt.cluster_dbscan(ac_df, eps=55)

In [ ]:
me3_nanodomain_df = hmt.calculate_nanodomain_characteristics(me3_df)
ac_nanodomain_df = hmt.calculate_nanodomain_characteristics(ac_df)

In [ ]:
me3_df[me3_df["cluster_label"]==649]

In [ ]:
plt.figure(figsize=(12,8))
plt.subplot(1, 2, 1)
plt.scatter(data=me3_df, x="x [nm]", y="y [nm]", c="cluster_color", s=1, linewidths=0)
plt.xlabel("x [nm]")
plt.ylabel("y [nm]")
plt.title("H3K27me3 Clusters")

plt.subplot(1, 2, 2)
plt.scatter(data=ac_df, x="x [nm]", y="y [nm]", c="cluster_color", s=1, linewidths=0)
plt.xlabel("x [nm]")
plt.ylabel("y [nm]")
plt.title("H3K27ac Clusters")

plt.tight_layout()
plt.show()

In [ ]:
hmt.plot_clusters_napari(me3_df, ac_df)
